## video sentiment

In [ ]:
import os
import tqdm
import cv2
import zipfile
import shutil
import subprocess

In [ ]:
import pandas as pd #著名数据处理包
import nltk 
from nltk import word_tokenize #分词函数
from nltk.corpus import stopwords #停止词表，如a,the等不重要的词
from nltk.corpus import sentiwordnet as swn #得到单词情感得分
import string #本文用它导入标点符号，如!"#$%& 

In [ ]:
#视频路径
video_path = './data/65814f7a0000000016004ad1.mp4'
video_id='65814f7a0000000016004ad1'
# 打开视频文件
cap = cv2.VideoCapture(video_path)
print(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)  # 获取视频的帧率
frame_interval = int(fps * 6)  # 每六秒提取一帧
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # 获取总帧数
extracted_frame_count = 0

print(f"Total frames to process: {total_frames // frame_interval}")

# 提取帧的过程
frame_count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 每6秒提取一帧
    if frame_count % frame_interval == 0:
        frame_path = os.path.join(frames_folder, f"{video_id[6:]}_{extracted_frame_count:04d}.jpg")
        cv2.imwrite(frame_path, frame)
        extracted_frame_count += 1

    frame_count += 1

cap.release()

# 生成临时地址的 TXT 文件，使用视频ID命名，存储绝对地址
txt_file_path = os.path.join(temp_folder, f"{video_id}.txt")
with open(txt_file_path, "w") as txt_file:
    for frame_name in os.listdir(frames_folder):
        absolute_frame_path = os.path.abspath(os.path.join(frames_folder, frame_name))  # 获取绝对路径
        txt_file.write(absolute_frame_path + "\n")

# 运行 sentibank.py 生成 JSON 文件
result = subprocess.run([
    "python", "sentibank.py",
    txt_file_path,
    "GPU",
    "DEVICE_ID=0"
], capture_output=True, text=True)  # 捕获输出和错误信息

# 检查返回码并打印输出
if result.returncode == 0:
    print("The script ran successfully.")
else:
    print(f"Error occurred while running the script: {result.stderr}")

In [ ]:
#举例  导入函数包
import pandas as pd #著名数据处理包
import nltk 
from nltk import word_tokenize #分词函数
from nltk.corpus import stopwords #停止词表，如a,the等不重要的词
from nltk.corpus import sentiwordnet as swn #得到单词情感得分
import string #本文用它导入标点符号，如!"#$%& 
import json
import math

In [ ]:
#经过上述处理视频得到的json文件
filename = r'./data/65814f7a0000000016004ad1.json'
with open(filename) as f:
    ANP=json.load(f)

In [ ]:
#计算视频情感列表
video_content = pd.DataFrame(columns=['video_id', 'senti_means'])

with open(filename) as f:
    ANP=json.load(f)

anp=[]
for i in range(len(ANP['images'])):
    anp.append(ANP['images'][i]['bi-concepts'])

for i in range(len(anp)):
    four={}
    j=0
    for k,v in anp[i].items():
        j+=1
        if j<=4:
            four[k]=v
    anp[i].clear()
    anp[i].update(four)

datall=pd.DataFrame(columns =['adj','score'])
for i in range(len(anp)):
    data=pd.DataFrame.from_dict(anp[i],orient='index',columns=['score'])
    data=data.reset_index().rename(columns={'index':'adj'})
    #print(data)
    datall=datall.append(data)
datall.reset_index(drop=True,inplace=True)

#增加几列数据
datall.insert(1,'noun','')
datall.insert(2,'as','')
datall.insert(3,'ns','')

#将adj与noun分开
for i in range(len(datall)):
    p=datall['adj'][i].split('_')
    datall['adj'][i]=p[0]
    datall['noun'][i]=p[1]

#计算形容词的情感值
for i in range(len(datall)):
    a = list(swn.senti_synsets(datall.iloc[i,0],'a'))
    #n = list(swn.senti_synsets(datall.iloc[i,1],'n'))
    s = 0
    ra = 0
    if len(a) > 0:
        for j in range(len(a)):
            s += (a[j].pos_score()-a[j].neg_score())/(j+1)
            ra += 1/(j+1)
        datall['as'][i]=s/ra
    else:
        datall['as'][i]=0

#计算名词的情感值
for i in range(len(datall)):
    #a = list(swn.senti_synsets(datall.iloc[i,0],'a'))
    n = list(swn.senti_synsets(datall.iloc[i,1],'n'))
    s = 0
    ra = 0
    if len(n) > 0:
        for j in range(len(n)):
            s += (n[j].pos_score()-n[j].neg_score())/(j+1)
            ra += 1/(j+1)
        datall['ns'][i]=s/ra
    else:
        datall['ns'][i]=0

senti=pd.DataFrame(columns =['senti'])
#math.log(val)/5+1  math.log(datall['score'][i])/5+1
k=0
for i in range(len(datall)):
    if i%4 <3:
        if datall['as'][i]*datall['ns'][i] >=0:
            k+=(datall['as'][i]+datall['ns'][i])*(math.log(datall['score'][i])/5+1)
        else:
            k+=datall['as'][i]*(math.log(datall['score'][i])/5+1)
    else:
        if datall['as'][i]*datall['ns'][i] >=0:
            k+=(datall['as'][i]+datall['ns'][i])*(math.log(datall['score'][i])/5+1)
        else:
            k+=datall['as'][i]*(math.log(datall['score'][i])/5+1)
        senti=senti.append({'senti':k},ignore_index=True)
        k=0
#senti['group'] = senti.index // 3  # 每3行一组
means = senti.groupby(senti.index // 3)['senti'].mean().tolist()

video_content = pd.concat([
    video_content,
    pd.DataFrame({'video_id': [video_id], 'senti_means': [means]})
], ignore_index=True)

In [ ]:
#计算音频和视频的情感一致性
def calculate_pearson_corr(audio_list, video_list):
    # 过滤非数值元素
    audio_vals = audio_list
    video_vals = video_list
    
    # 校验最小数据点数量
    if len(audio_vals) < 2 or len(video_vals) < 2:
        return np.nan
    
    # 按最短列表长度截断，保证两个列表长度一致
    min_len = min(len(audio_vals), len(video_vals))
    audio_trimmed = audio_vals[:min_len]
    video_trimmed = video_vals[:min_len]
    
    # 计算相关系数
    corr, _ = pearsonr(audio_trimmed, video_trimmed)
    return corr

In [ ]:
merged=pd.dataframe()

In [ ]:
consistency

In [ ]:
#audio_list替换为Audio sentiment score & Audio arousal score.ipynb中得到的Audio sentiment score
merged['Audio-Vudio_valance_congruency'] = calculate_pearson_corr(audio_list, video_content['senti_means'])

In [ ]:
#text_valance_mean替换为 Text sentiment score & Text arousal score.ipynb中得到的Text sentiment score
merged['Text-audio_valance_congruency'] = 1 - (text_valance_mean - mean(audio_list)).abs()

In [ ]:

merged['Text-Vudio_valance_congruency'] = 1 - (text_valance_mean - mean(video_content['senti_means'])).abs()